# Fine-tuning de Modelo de Series Temporales con QLoRA (Kaggle)
Notebook adaptado para Kaggle, usa una sola GPU (T4) para evitar conflictos con DataParallel y QLoRA.

In [1]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 15.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import json
import zipfile
import shutil
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Forzar uso de una sola GPU (la 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Nombre GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU disponible: True
Nombre GPU: Tesla T4
Memoria total: 15.64 GB


In [3]:
# Celda 3: Localizar modelo y dataset automáticamente
input_dir = "/kaggle/input"

# --- 1. Buscar el modelo (carpeta con config.json) ---
teacher_path = None
for root, dirs, files in os.walk(input_dir):
    if "config.json" in files:
        teacher_path = root
        break

if teacher_path is None:
    raise FileNotFoundError("No se encontró carpeta con config.json en /kaggle/input")
print(f"✅ Modelo localizado en: {teacher_path}")

# --- 2. Buscar el dataset TS_Dataset.jsonl ---
jsonl_path = None
for root, dirs, files in os.walk(input_dir):
    for f in files:
        if f == "TS_Dataset.jsonl" or (f.endswith(".jsonl") and "TS_Dataset" in f):
            jsonl_path = os.path.join(root, f)
            break
    if jsonl_path:
        break

if jsonl_path is None:
    raise FileNotFoundError("No se encontró TS_Dataset.jsonl en /kaggle/input")
print(f"✅ Dataset localizado en: {jsonl_path}")
JSONL_PATH = jsonl_path

✅ Modelo localizado en: /kaggle/input/models/danielrangelsanch/profesor/other/default/1/profesor
✅ Dataset localizado en: /kaggle/input/datasets/danielrangelsanch/ts-dataset-jsonl/TS_Dataset.jsonl


In [4]:
# Celda 4: Cargar dataset
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    if content.startswith('['):
        return json.loads(content)
    else:
        return [json.loads(line) for line in content.split('\n') if line.strip()]

data = load_jsonl(JSONL_PATH)
print(f"Total de ejemplos: {len(data)}")

dataset = Dataset.from_list(data)

Total de ejemplos: 8787


In [5]:
# Celda 5: Tokenización con padding fijo
tokenizer = AutoTokenizer.from_pretrained(teacher_path)
tokenizer.pad_token = tokenizer.eos_token

max_length = 1024  # Ajusta si es necesario

def format_example(example):
    series_str = ", ".join([f"{x:.4f}" for x in example["series"]])
    meta = example.get("metadata", {})
    meta_str = f"unidades={meta.get('units','')}, frecuencia={meta.get('frequency','')}, inicio={meta.get('start','')}, fin={meta.get('end','')}"
    prompt = (
        "### Instrucción: Describe la siguiente serie temporal y enumera sus características principales.\n"
        f"### Serie: {series_str}\n"
        f"### Metadatos: {meta_str}\n"
        "### Respuesta:\n"
    )
    target = f"{example['description']}\n### Características:\n{example['characteristics']}"
    return prompt + target

def tokenize_function(example):
    text = format_example(example)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"]
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names,
)

# Convertir a tensores de PyTorch
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/8787 [00:00<?, ? examples/s]

In [6]:
# Celda 6: Configurar QLoRA y LoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    teacher_path,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},   # Forzar una sola GPU
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Determinar target_modules automáticamente
linear_modules = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_modules.add(name.split('.')[-1])

possible_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
target_modules = [m for m in possible_targets if m in linear_modules]
if not target_modules:
    target_modules = list(linear_modules - {"lm_head"})
print(f"Target modules: {target_modules}")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [7]:
# Celda 7: Entrenamiento
output_dir = "/kaggle/working/teacher-lora-adapters"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    max_grad_norm=0.3,
    warmup_steps=10,               # Reemplaza warmup_ratio (deprecado)
    lr_scheduler_type="cosine",
    report_to="none",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    ddp_find_unused_parameters=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    # No data_collator porque el dataset ya está en tensores
)

torch.cuda.empty_cache()
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.051820
20,1.084201
30,1.077233
40,1.024739
50,1.126848
60,0.944220
70,0.994964
80,0.990190
90,1.016297
100,1.084577


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=4394, training_loss=1.0341179106091172, metrics={'train_runtime': 13204.2526, 'train_samples_per_second': 1.331, 'train_steps_per_second': 0.333, 'total_flos': 3.959390900846592e+16, 'train_loss': 1.0341179106091172, 'epoch': 2.0})

In [8]:
# Celda 8: Guardar adaptador y tokenizer, comprimir
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en {output_dir}")

zip_output = "/kaggle/working/teacher-lora-adapters.zip"
shutil.make_archive(zip_output.replace('.zip',''), 'zip', output_dir)
print(f"Adaptador comprimido en: {zip_output}")

Modelo guardado en /kaggle/working/teacher-lora-adapters
Adaptador comprimido en: /kaggle/working/teacher-lora-adapters.zip


In [9]:
# Celda 9: Enlace de descarga
from IPython.display import FileLink
FileLink(zip_output)

/kaggle/working/teacher-lora-adapters.zip

In [10]:
# Celda 10 (Opcional): Inferencia de prueba
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    teacher_path,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},
    trust_remote_code=True,
)
model_inf = PeftModel.from_pretrained(base_model, output_dir)
model_inf.eval()

def generate_description(series, metadata=None):
    series_str = ", ".join([f"{x:.4f}" for x in series])
    meta_str = ""
    if metadata:
        meta_str = f"unidades={metadata.get('units','')}, frecuencia={metadata.get('frequency','')}, inicio={metadata.get('start','')}, fin={metadata.get('end','')}"
    prompt = (
        "### Instrucción: Describe la siguiente serie temporal y enumera sus características principales.\n"
        f"### Serie: {series_str}\n"
        f"### Metadatos: {meta_str}\n"
        "### Respuesta:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model_inf.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

ejemplo = data[0]
desc = generate_description(ejemplo["series"], ejemplo["metadata"])
print(desc)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Instrucción: Describe la siguiente serie temporal y enumera sus características principales.
### Serie: 20.8820, 20.2001, 20.4894, 21.1204, 20.9338, 19.5114, 20.4750, 19.9243, 19.9484, 20.2053, 20.0720, 20.7271, 20.3805, 20.0608, 20.2219, 20.1668, 20.7470, 19.8974, 20.1565, 19.5730, 18.7235, 20.3268, 20.4322, 19.6289, 21.1349, 19.2728, 20.0229, 19.9064, 20.7664, 20.7347, 20.0775, 20.1891, 19.5561, 19.0096, 19.8260, 20.0782, 20.6151, 20.6012, 19.8063, 19.8488, 19.4757, 19.2900, 19.1469, 20.9754, 19.7452, 19.7810, 19.3736, 20.3887, 19.1931, 19.8936, 19.5523, 20.1935, 19.7446, 19.4097, 19.9859, 20.2142, 20.0333, 20.1512, 19.6828, 19.8186, 19.6638, 19.8202, 19.5934, 19.1369, 20.0887, 19.7991, 19.1849, 20.2314, 19.5464, 20.0260, 20.3645, 20.0645, 20.5697, 19.3826, 20.2012, 19.6576, 19.5646, 19.7106, 19.8442, 20.0281, 19.4174, 20.4504, 20.2328, 19.2319, 20.7441, 20.9479, 20.5894, 19.9100, 19.4646, 20.5272, 19.7984, 20.6112, 20.1041, 20.4883, 20.1782, 20.3533, 20.0053, 20.8929, 20.0635, 2